# Week 6 Student Worksheet: Spatial Prediction Shootout

## Traditional Stats vs. Machine Learning: Filling the Gaps

> *"Kriging tells you where it's uncertain. ML just says 'trust me.'"*

### Today's Mission

Transform Week 5's discrete rainfall stations into **continuous rainfall surfaces** using **two fundamentally different approaches**:

1. **Kriging (Statistical)** — uses spatial correlation + provides uncertainty
2. **Random Forest (ML)** — uses data patterns + easily adds features

Then compare them head-to-head with two simpler methods (Nearest Neighbor, IDW) and determine which gives the Commander more actionable intelligence.

> Fill in the code cells marked with `# YOUR CODE HERE`. Use AI tools strategically — but understand every line you write.

In [ ]:
# Install required packages (run once)
%pip install pykrige scikit-learn rasterio rasterstats --quiet

---

## Cell [1]: Environment Setup & Data Loading (Slide 2)

Load the Fung-wong typhoon data and filter to the study area. Reuse your Week 5 `parse_rainfall_json()` function.

**AI Prompt Suggestion**:
```
I need to load fungwong_202511.json and parse it into a GeoDataFrame using
my Week 5 parse_rainfall_json() function. Then filter to 花蓮縣 and 宜蘭縣,
remove -998/zero rain stations, and convert to EPSG:3826.
Show me the top 5 stations by rain_1hr.
```

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from shapely.geometry import Point
warnings.filterwarnings('ignore')

# ── Helper: parse CWA rainfall JSON → GeoDataFrame ──────────────────
def parse_rainfall_json(filepath):
    """Parse CWA O-A0002-001 JSON into a GeoDataFrame."""
    with open(filepath, encoding='utf-8') as f:
        data = json.load(f)

    rows = []
    for st in data['records']['Station']:
        geo = st['GeoInfo']
        coord = geo['Coordinates'][0]
        rain = st['RainfallElement']
        rows.append({
            'station_name': st['StationName'],
            'station_id':   st['StationId'],
            'county':       geo['CountyName'],
            'town':         geo['TownName'],
            'lat':          coord['StationLatitude'],
            'lon':          coord['StationLongitude'],
            'rain_10min':   rain['Past10Min']['Precipitation'],
            'rain_1hr':     rain['Past1hr']['Precipitation'],
            'rain_3hr':     rain['Past3hr']['Precipitation'],
            'rain_6hr':     rain['Past6hr']['Precipitation'],
            'rain_12hr':    rain['Past12hr']['Precipitation'],
            'rain_24hr':    rain['Past24hr']['Precipitation'],
        })

    df = pd.DataFrame(rows)
    geometry = [Point(lon, lat) for lon, lat in zip(df.lon, df.lat)]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
    return gdf

# ── Load & filter ────────────────────────────────────────────────────
rain_gdf = parse_rainfall_json('data/scenarios/fungwong_202511.json')

# Filter: 花蓮縣 + 宜蘭縣, remove missing (-998) and zero rain
study_rain = rain_gdf[rain_gdf['county'].isin(['花蓮縣', '宜蘭縣'])].copy()
study_rain = study_rain[study_rain['rain_1hr'] > 0].copy()

# Convert to EPSG:3826 (TWD97 / TM2, meters)
study_rain_3826 = study_rain.to_crs(epsg=3826)

# Extract coordinate arrays for Kriging / ML
x = study_rain_3826.geometry.x.values  # Easting (meters)
y = study_rain_3826.geometry.y.values  # Northing (meters)
z = study_rain_3826['rain_1hr'].values

print(f"Study area stations (rain > 0): {len(study_rain_3826)}")
print(f"CRS: {study_rain_3826.crs}")
print(f"\nTop 5 stations:")
print(study_rain_3826.nlargest(5, 'rain_1hr')[['station_name', 'county', 'rain_1hr']].to_string(index=False))

---

## Cell [2a]: Variogram — First Attempt (Naive)

Try running Kriging directly on the raw rainfall data. **Don't worry if it looks bad — that's the point.**

**Key Concepts** (will make sense after you see the result):
- **Nugget**: How noisy is each measurement?
- **Sill**: What's the maximum difference between stations?
- **Range**: How far apart do stations need to be before they stop "knowing about each other"?

**CRS Warning**: The x, y arrays MUST be in EPSG:3826 (meters). Using lat/lon will give wrong results.

In [ ]:
from pykrige.ok import OrdinaryKriging

# 🔴 First attempt: Kriging on raw rainfall data
initial_sill = float(z.var())
initial_range = 50000.0
initial_nugget = float(z.var() * 0.1)

OK_naive = OrdinaryKriging(x, y, z, variogram_model='spherical',
                            verbose=False, enable_plotting=True, nlags=15,
                            variogram_parameters={'sill': initial_sill,
                                                  'range': initial_range,
                                                  'nugget': initial_nugget})

params = OK_naive.variogram_model_parameters
print(f"Sill:   {params[0]:.1f}")
print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
print(f"Nugget: {params[2]:.1f}")
print("\n⚠️ Look at the plot — do the dots follow the curve?")

## Cell [2b]: Why Did It Fail? — Look at the Histogram

The variogram looked bad. Before blaming the tool, **look at your data**.

Plot a histogram of the rainfall values. What do you see?

In [ ]:
# Histogram: raw vs log-transform
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(z, bins=30, color='tomato', edgecolor='black')
axes[0].set_title('Raw Rainfall (mm/hr)')
axes[0].set_xlabel('Rainfall (mm/hr)')

z_log = np.log1p(z)
axes[1].hist(z_log, bins=30, color='steelblue', edgecolor='black')
axes[1].set_title('After log(1+z)')
axes[1].set_xlabel('log(1 + Rainfall)')
plt.tight_layout()
plt.show()

print("Left: most stations < 10 mm, but a few are 50-130 mm.")
print("Those extreme values mess up the variogram.")
print("Right: after log-transform, the values are more balanced.")

## Cell [2c]: Variogram — Second Attempt (with Log-Transform)

Now redo the variogram using the log-transformed data. Compare this plot with Cell [2a] — it should be much better.

**Rule of thumb**: If your histogram has a long tail to the right, apply `np.log1p(z)` before Kriging, then use `np.expm1()` to convert back after prediction.

In [ ]:
# 🟢 Second attempt: Kriging on log-transformed data
z_log = np.log1p(z)

initial_sill = float(z_log.var())
initial_range = 50000.0
initial_nugget = float(z_log.var() * 0.1)

OK = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
                      verbose=False, enable_plotting=True, nlags=15,
                      variogram_parameters={'sill': initial_sill,
                                            'range': initial_range,
                                            'nugget': initial_nugget})

params = OK.variogram_model_parameters
print(f"Sill:   {params[0]:.3f}")
print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
print(f"Nugget: {params[2]:.3f}")
print("\n✅ Compare with Cell [2a] — the dots should follow the curve now.")

## Cell [2d]: Which Variogram Fits Best? — Range & Model Comparison (Slide 7)

Compare variograms **one variable at a time** — this is how scientists isolate effects:

- **Figure 1**: Fix model = **Spherical**, vary Range (50km, 25km, 15km) → what does Range do?
- **Figure 2**: Fix model = **Exponential**, vary Range (50km, 25km, 15km) → same exercise
- **Then compare**: same Range, different model → what does the model choice do?

**AI Prompt Suggestion**:
```
Create two sets of variogram plots using z_log data:
Figure 1: Spherical model with Range = 50km, 25km, 15km (1×3 subplots)
Figure 2: Exponential model with Range = 50km, 25km, 15km (1×3 subplots)
For each, plot empirical variogram (ok.lags vs ok.semivariance) as red dots
and the fitted model curve as a black line.
Compute SSE for each config and print a summary table.
Compare within each model first (Range effect), then across models (Model effect).
```

In [ ]:
ranges_m = [50000, 25000, 15000]
ranges_km = [50, 25, 15]
models = ['spherical', 'exponential']
results = []

fig, all_axes = plt.subplots(2, 3, figsize=(18, 10))

for row, model in enumerate(models):
    for col, (rkm, rm) in enumerate(zip(ranges_km, ranges_m)):
        ax = all_axes[row, col]
        sill_val = float(z_log.var())
        nug_val = float(z_log.var() * 0.1)

        ok_test = OrdinaryKriging(x, y, z_log, variogram_model=model,
                                   verbose=False, enable_plotting=False, nlags=15,
                                   variogram_parameters={'sill': sill_val,
                                                         'range': rm,
                                                         'nugget': nug_val})
        # Empirical variogram
        ax.scatter(ok_test.lags / 1000, ok_test.semivariance, c='red',
                   s=40, zorder=5, label='Empirical')

        # Fitted model curve
        lag_fine = np.linspace(0, ok_test.lags.max(), 200)
        sv_fine = ok_test.variogram_function(ok_test.variogram_model_parameters, lag_fine)
        ax.plot(lag_fine / 1000, sv_fine, 'k-', lw=2, label='Fitted')

        # SSE
        sv_pred = ok_test.variogram_function(ok_test.variogram_model_parameters, ok_test.lags)
        sse = np.sum((ok_test.semivariance - sv_pred) ** 2)

        ax.set_title(f'{model.title()} | Range={rkm}km\nSSE={sse:.4f}', fontsize=11)
        ax.set_xlabel('Lag (km)')
        ax.set_ylabel('Semivariance')
        ax.legend(fontsize=8)

        p = ok_test.variogram_model_parameters
        results.append({'Model': model, 'Init_Range_km': rkm,
                        'Fit_Sill': p[0], 'Fit_Range_km': p[1]/1000, 'Fit_Nugget': p[2],
                        'SSE': sse})

plt.suptitle('Variogram Model Comparison: Spherical vs Exponential × 3 Ranges', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False, float_format='%.4f'))
print(f"\n✅ Best fit: {df_results.loc[df_results['SSE'].idxmin(), 'Model']} "
      f"with init Range={df_results.loc[df_results['SSE'].idxmin(), 'Init_Range_km']}km "
      f"(SSE={df_results['SSE'].min():.4f})")

## Cell [3]: Define the Interpolation Grid & Run Kriging (Slide 8)

Create a regular grid covering the study area with 1000m resolution, then execute Kriging.

**Think about**:
- What units is the grid in? (Hint: same as your CRS = meters)
- Why add a buffer around the station extent?

**Important**: Kriging runs in log-space. After prediction, **back-transform** with `np.expm1()` to get rainfall in mm/hr.

In [ ]:
import time

buffer_m = 5000
resolution = 1000  # meters

x_min = x.min() - buffer_m
x_max = x.max() + buffer_m
y_min = y.min() - buffer_m
y_max = y.max() + buffer_m
grid_x = np.arange(x_min, x_max, resolution)
grid_y = np.arange(y_min, y_max, resolution)

print(f"Grid: {len(grid_x)}×{len(grid_y)} = {len(grid_x)*len(grid_y):,} points @ {resolution}m")

t0 = time.time()
z_kriging_log, ss_kriging_log = OK.execute('grid', grid_x, grid_y)
print(f"✓ Kriging (log-space) done in {time.time()-t0:.1f}s")

# Back-transform to real rainfall (mm/hr)
z_kriging = np.expm1(z_kriging_log)
z_kriging[z_kriging < 0] = 0
ss_kriging = ss_kriging_log  # keep log-space variance for Sigma Map

print(f"  z range (mm/hr): {np.nanmin(z_kriging):.1f} - {np.nanmax(z_kriging):.1f}")

---

## Cell [4]: Machine Learning — Random Forest Prediction (Slide 9)

**Captain's Log**: Treating coordinates as input features. ML learns `f(easting, northing) → rainfall`. Simple, but no uncertainty map.

**AI Prompt Suggestion**:
```
Train a RandomForestRegressor from scikit-learn using [easting, northing]
as features (X_train) and rain_1hr as target (y_train).
Use n_estimators=200, min_samples_leaf=3, random_state=42.
Then predict on the same grid as Kriging.
```

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X_train = np.column_stack([x, y])
y_train = z

rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)
rf.fit(X_train, y_train)
print(f"RF Training R²: {rf.score(X_train, y_train):.3f}")

grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)
X_grid = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])

t0 = time.time()
z_rf = rf.predict(X_grid).reshape(grid_xx.shape)
print(f"✓ Random Forest done in {time.time()-t0:.1f}s")
print(f"  z range: {z_rf.min():.1f} - {z_rf.max():.1f} mm/hr")

## Cell [5]: ML Glass Box — Feature Importance (Slide 11)

**Captain's Question**: "AI used what to predict floods — latitude or elevation?"

Even though ML is a "black box", we can peek inside with `.feature_importances_`.

In [ ]:
importances = rf.feature_importances_
print("Feature Importance:")
print(f"  Easting (X):  {importances[0]:.3f}")
print(f"  Northing (Y): {importances[1]:.3f}")
print(f"\nThe model relies mostly on {'easting' if importances[0] > importances[1] else 'northing'}.")
print("Think: does this make physical sense for Typhoon Fung-wong?")

---

## ⏸️ Lab 1: The Four-Way Interpolation Shootout (40 min)

### Cell [6]: Nearest Neighbor + IDW

Compute two additional interpolation methods so we can compare all four.

**AI Prompt Suggestion**:
```
Use scipy.interpolate.NearestNDInterpolator for nearest neighbor interpolation,
and manual IDW with scipy.spatial.distance.cdist (power=2) for IDW.
Both should predict on the same meshgrid as Kriging and RF.
Note: Don't use Rbf(function='inverse') — it's not real IDW and produces extreme values.
```

In [ ]:
from scipy.interpolate import NearestNDInterpolator
from scipy.spatial.distance import cdist

# Nearest Neighbor
nn_interp = NearestNDInterpolator(list(zip(x, y)), z)
z_nn = nn_interp(grid_xx, grid_yy)

# IDW (manual, power=2)
pts = np.column_stack([x, y])
grid_pts = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])
dists = cdist(grid_pts, pts)
dists[dists < 1] = 1  # avoid division by zero
power = 2
weights = 1.0 / (dists ** power)
z_idw = ((weights @ z) / weights.sum(axis=1)).reshape(grid_xx.shape)

print("✓ Nearest Neighbor + IDW computed")

### Cell [7]: Four Methods Side by Side (Slide 13)

Create a 2×2 figure comparing all four interpolation methods.

**AI Prompt Suggestion**:
```
Create a 2×2 matplotlib figure comparing four interpolation results:
- Nearest Neighbor (z_nn)
- IDW (z_idw)
- Ordinary Kriging (z_kriging)
- Random Forest (z_rf)
Use YlOrRd colormap, same vmin/vmax, overlay station points in black.
Add descriptive subtitles (e.g., "Voronoi Patchwork", "Bullseye Effect",
"Smooth + Sigma Map", "ML Block Artifacts").
Save as 'interpolation_shootout.png'.
```

In [ ]:
vmax = max(z) * 1.1
methods = [
    ('Nearest Neighbor\n(Voronoi / "Patchwork")', z_nn),
    ('IDW\n(Bullseye Effect)', z_idw),
    ('Ordinary Kriging\n(Smooth + Sigma Map)', z_kriging),
    ('Random Forest\n(ML "Block" Artifacts)', z_rf),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
for ax, (title, data) in zip(axes.flatten(), methods):
    im = ax.imshow(data, extent=[x_min, x_max, y_min, y_max],
                   origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
    ax.scatter(x, y, c='black', s=8, zorder=5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.7, label='mm/hr')

plt.suptitle('Typhoon Fung-wong — Four Interpolation Methods Compared', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('interpolation_shootout.png', dpi=150, bbox_inches='tight')
plt.show()

### Cell [8]: Kriging vs RF — Direct Comparison

Create a 3-panel comparison: Kriging | Random Forest | Difference Map

The difference map reveals **where the two methods disagree** — these are the areas where the Commander needs extra caution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Kriging
im1 = axes[0].imshow(z_kriging, extent=[x_min, x_max, y_min, y_max],
                      origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
axes[0].scatter(x, y, c='black', s=10, zorder=5)
axes[0].set_title('Ordinary Kriging', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=axes[0], shrink=0.8, label='mm/hr')

# Random Forest
im2 = axes[1].imshow(z_rf, extent=[x_min, x_max, y_min, y_max],
                      origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
axes[1].scatter(x, y, c='black', s=10, zorder=5)
axes[1].set_title('Random Forest', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[1], shrink=0.8, label='mm/hr')

# Difference
diff = z_kriging - z_rf
vlim = max(abs(np.nanmin(diff)), abs(np.nanmax(diff)))
im3 = axes[2].imshow(diff, extent=[x_min, x_max, y_min, y_max],
                      origin='lower', cmap='RdBu_r', vmin=-vlim, vmax=vlim)
axes[2].scatter(x, y, c='black', s=10, zorder=5)
axes[2].set_title('Difference (Kriging − RF)', fontsize=12, fontweight='bold')
plt.colorbar(im3, ax=axes[2], shrink=0.8, label='mm/hr')

plt.suptitle('Kriging vs Random Forest — Direct Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('kriging_vs_rf.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean absolute difference: {np.nanmean(np.abs(diff)):.1f} mm/hr")
print(f"Max difference: {np.nanmax(np.abs(diff)):.1f} mm/hr")

### Lab 1 Reflection

**Questions to answer** (write in the cell below):

1. Which method produces the most physically realistic rainfall pattern? Why?
2. Where do Kriging and RF disagree the most? What does this tell you?
3. What "artifacts" do you observe in the NN and RF results?
4. If you were the Commander, which method would you trust for evacuation decisions? Why?

**Your Lab 1 reflection here:**

1. **Most realistic**: Kriging — it produces smooth gradients that match how rainfall actually varies in space, and doesn't create abrupt boundaries like NN or blocky artifacts like RF.

2. **Disagreement areas**: Kriging and RF diverge most in mountainous areas far from stations. Kriging smoothly extrapolates toward the global mean, while RF replicates the nearest training pattern, creating flat patches.

3. **Artifacts**: NN shows sharp Voronoi polygon edges (each pixel just copies the nearest station). RF shows rectangular step-like blocks — the decision tree splits on coordinate thresholds, producing a "pixelated" look.

4. **Commander's choice**: Kriging — because it provides a confidence map (sigma). The Commander can distinguish "high rain, confident" from "high rain, uncertain" and allocate resources accordingly. RF just gives a number with no reliability estimate.

---

## ⏸️ Lab 2: Confidence & Uncertainty Diagnosis (30 min)

### Cell [9]: The Sigma Map — Kriging's Unique Weapon (Slide 15)

**Captain's Log**: This is what makes Kriging special. No other method natively provides a confidence map.

- `ss_kriging` = Kriging variance at each grid point
- Low variance → many nearby stations → reliable estimate
- High variance → far from stations → uncertain

**For the Commander**:
- HIGH rainfall + LOW variance = CONFIRMED: Evacuate now
- HIGH rainfall + HIGH variance = UNCERTAIN: Deploy sensors first

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Rainfall estimate
im1 = axes[0].imshow(z_kriging, extent=[x_min, x_max, y_min, y_max],
                      origin='lower', cmap='YlOrRd', vmin=0)
axes[0].scatter(x, y, c='black', s=10, zorder=5)
axes[0].set_title('Estimated Rainfall (mm/hr)')
plt.colorbar(im1, ax=axes[0], shrink=0.8, label='mm/hr')

# Kriging Variance (Sigma Map)
im2 = axes[1].imshow(ss_kriging, extent=[x_min, x_max, y_min, y_max],
                      origin='lower', cmap='Blues', vmin=0)
axes[1].scatter(x, y, c='red', s=10, zorder=5, label='Stations')
axes[1].set_title('Kriging Sigma Map (Uncertainty)')
axes[1].legend(loc='upper right')
plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Variance')

plt.suptitle('The Honest Report Card: Estimate + Confidence', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('sigma_map.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Variance range: {np.nanmin(ss_kriging):.3f} - {np.nanmax(ss_kriging):.3f}")

### Cell [9b]: Nugget Effect — Why Extreme Rain Gets Diluted (Slide 7)

Suao recorded **130.5 mm/hr**, but default Kriging predicts only ~71 mm at 500m away. Why?

**Nugget** controls how much the model trusts the actual measurements:
- High Nugget = "measurements are noisy" → smooths everything → extreme values diluted
- Low Nugget = "measurements are accurate" → passes through stations → extremes preserved

**Task**: Compare Nugget=10% vs Nugget=1% on a zoomed-in map around Suao. Which preserves the extreme rainfall better?

**AI Prompt Suggestion**:
```
Create two OrdinaryKriging models on z_log with identical parameters except
nugget: one with nugget = sill * 0.10, one with nugget = sill * 0.01.
Predict on a 20km×20km grid centered on the station with maximum rainfall.
Show side-by-side maps and print predicted values at 0m, 500m, 1000m, 2000m
from that station. Which Nugget preserves the extreme value better?
```

In [ ]:
# Find station with max rainfall
suao_idx = np.argmax(z)
suao_x, suao_y, suao_z = x[suao_idx], y[suao_idx], z[suao_idx]
suao_name = study_rain_3826.iloc[suao_idx]['station_name']
print(f"Max rainfall station: {suao_name} = {suao_z:.1f} mm/hr")

sill_val = float(z_log.var())

# Two Kriging models: high nugget (10%) vs low nugget (1%)
configs = [
    ('Nugget = 10% sill', sill_val * 0.10),
    ('Nugget = 1% sill',  sill_val * 0.01),
]

# Local grid: 20km box around max station
local_buf = 10000
lx = np.arange(suao_x - local_buf, suao_x + local_buf, 500)
ly = np.arange(suao_y - local_buf, suao_y + local_buf, 500)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
offsets = [0, 500, 1000, 2000]

for ax, (label, nug) in zip(axes, configs):
    ok_nug = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
                              verbose=False, enable_plotting=False, nlags=15,
                              variogram_parameters={'sill': sill_val,
                                                    'range': 50000.0,
                                                    'nugget': nug})
    z_local_log, _ = ok_nug.execute('grid', lx, ly)
    z_local = np.expm1(z_local_log)
    z_local[z_local < 0] = 0

    im = ax.imshow(z_local, extent=[lx[0], lx[-1], ly[0], ly[-1]],
                   origin='lower', cmap='YlOrRd', vmin=0, vmax=suao_z * 1.1)
    ax.scatter([suao_x], [suao_y], c='blue', s=80, marker='*', zorder=5, label=f'{suao_name}')
    ax.set_title(f'{label}\nObs={suao_z:.1f} mm/hr', fontsize=12, fontweight='bold')
    ax.legend()
    plt.colorbar(im, ax=ax, shrink=0.8, label='mm/hr')

    # Print predictions at offsets
    print(f"\n{label}:")
    for d in offsets:
        pred_log, _ = ok_nug.execute('points', np.array([suao_x + d]),
                                      np.array([suao_y]))
        pred = float(np.expm1(pred_log))
        print(f"  {d:5d}m from station: {pred:.1f} mm/hr ({pred/suao_z*100:.0f}% of obs)")

plt.suptitle(f'Nugget Effect — Zoomed on {suao_name}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n🔑 Low nugget preserves extreme values better — appropriate for CWA calibrated stations.")

### Cell [10]: Export to GeoTIFF (Slide 16)

⚠️ **Flip warning**: `z_kriging` row 0 = south (numpy convention). GeoTIFF row 0 = north. Use `np.flipud()` to fix.

**AI Prompt Suggestion**:
```
Save 2D numpy arrays as GeoTIFF using rasterio. I need:
- from_bounds transform with my grid extent (x_min, y_min, x_max, y_max)
- CRS = EPSG:3826, dtype = float32
- Apply np.flipud() before writing
- Save kriging_rainfall.tif, kriging_variance.tif, and rf_rainfall.tif
```

In [ ]:
import rasterio
from rasterio.transform import from_bounds

transform = from_bounds(x_min, y_min, x_max, y_max,
                        width=z_kriging.shape[1], height=z_kriging.shape[0])

def save_geotiff(data, filename, crs='EPSG:3826'):
    data_flipped = np.flipud(data).astype(np.float32)
    with rasterio.open(filename, 'w', driver='GTiff',
        height=data_flipped.shape[0], width=data_flipped.shape[1],
        count=1, dtype='float32', crs=crs, transform=transform, nodata=-9999
    ) as dst:
        dst.write(data_flipped, 1)
    print(f"✓ Saved {filename}")

save_geotiff(z_kriging, 'kriging_rainfall.tif')
save_geotiff(ss_kriging, 'kriging_variance.tif')
save_geotiff(z_rf, 'rf_rainfall.tif')

### Cell [11]: Zonal Statistics — Township Decision Table

Compute per-township statistics from your Kriging and RF rasters, then compare them side-by-side.

**Required output**: A DataFrame with: 鄉鎮 | Kriging平均 | Kriging最大 | RF平均 | 平均variance | 可信度

In [ ]:
from rasterstats import zonal_stats

# Load township boundaries
towns = gpd.read_file('data/township/TOWN_MOI_1140318.shp')
study_towns = towns[towns['COUNTYNAME'].isin(['花蓮縣', '宜蘭縣'])].copy()
study_towns = study_towns.to_crs(epsg=3826)

# Zonal stats for each raster
krig_stats = zonal_stats(study_towns, 'kriging_rainfall.tif', stats=['mean', 'max'])
var_stats  = zonal_stats(study_towns, 'kriging_variance.tif', stats=['mean'])
rf_stats   = zonal_stats(study_towns, 'rf_rainfall.tif', stats=['mean'])

# Build summary table
summary = study_towns[['TOWNNAME', 'COUNTYNAME']].copy()
summary.columns = ['鄉鎮', '縣市']
summary['Kriging平均'] = [s['mean'] for s in krig_stats]
summary['Kriging最大'] = [s['max'] for s in krig_stats]
summary['RF平均']      = [s['mean'] for s in rf_stats]
summary['平均variance'] = [s['mean'] for s in var_stats]

# Drop rows with no data (townships outside raster extent)
summary = summary.dropna(subset=['Kriging平均'])

# Confidence classification
q33 = summary['平均variance'].quantile(0.33)
q66 = summary['平均variance'].quantile(0.66)
summary['可信度'] = pd.cut(summary['平均variance'],
                          bins=[-np.inf, q33, q66, np.inf],
                          labels=['HIGH', 'MEDIUM', 'LOW'])

summary = summary.sort_values('Kriging平均', ascending=False)
print(summary.to_string(index=False, float_format='%.1f'))
print(f"\n⚠️ HIGH rainfall + LOW confidence townships need extra sensor deployment.")

### Cell [12]: Thinking Challenge — Why Can't ML Give You a Sigma Map?

**Discussion Questions** (answer in the cell below):

1. Why does Kriging **natively** produce a variance map, but Random Forest does not?
2. Could you approximate uncertainty from RF using bootstrap or tree variance? What are the limitations?
3. In your zonal stats table, which townships show **HIGH rainfall + LOW confidence**? What should the Commander do about those?

**Your Lab 2 reflection here:**

1. **Kriging vs ML uncertainty**: Kriging solves a system of equations that minimizes estimation variance — the variance is a direct mathematical byproduct of the solution. RF averages votes from independent trees; there's no built-in mechanism to quantify how "sure" the ensemble is.

2. **ML uncertainty approximation**: Yes — you can compute the standard deviation across individual tree predictions as a proxy for uncertainty. However, this only captures model disagreement, not the theoretically optimal variance that Kriging provides. It also tends to underestimate uncertainty in extrapolation regions.

3. **High rain + low confidence townships**: These are mountain townships with few or no stations (e.g., 秀林鄉). The Commander should deploy mobile rain gauges or use radar data to validate before committing evacuation resources.

### Cell [13]: (Bonus) AI Advisor Consultation

Ask an AI model (Gemini, ChatGPT, or Claude):

> 「在花蓮山區，測站密度約 1 站 / 50 km²。我用 Kriging 和 Random Forest 分別做了降雨內插，結果在山區差異很大。Kriging 的 variance 在山區也很高。請問：(1) 我應該信哪個結果？(2) 如何改善山區的預測品質？」

Paste the AI's response below and write your own commentary.

**AI Response:**

(Skip — will paste after consulting an AI tool in class)

**My Commentary:**

Both methods struggle in data-sparse mountain areas, but for different reasons: Kriging reverts to the mean (conservative), RF memorizes the nearest pattern (potentially misleading). The practical solution is not better algorithms but better data — adding elevation as a co-variable (Co-Kriging or RF with DEM features) would improve both methods significantly.

---

## ARIA Evolution Recap

| Version | Week | Capability | Key Question |
|---------|------|-----------|---------------|
| v1.0 | W3 | River buffer + shelters | How close to the river? |
| v2.0 | W4 | + DEM terrain analysis | How steep is the terrain? |
| v3.0 | W5 | + Real-time rainfall stations | How much rain RIGHT NOW? |
| **v3.5** | **W6** | **+ Kriging & ML interpolation** | **What about areas with NO station? How confident are we?** |
| v4.0 | Final Project | Your extension! | Your question! |

---

*"Interpolation is not just filling space; it is predicting risk where sensors cannot reach."*